In [1]:
!pip install --no-index --find-links /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets/packages segmentation-models-pytorch albumentations omegaconf torchmetrics

Looking in links: /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets/packages
Processing /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets/packages/segmentation_models_pytorch-0.5.0-py3-none-any.whl


In [2]:
# ============================================================
# CONFIG — the only cell that differs between WildUAV notebooks
# ============================================================
import time
SESSION_START = time.time()

NOTEBOOK     = 'WU3_wilduav_s2026'
LOSSES       = ['ce', 'ced', 'cefd', 'adaptive']      # trained one after another in this session
SEED         = 2026
MODEL_NAME   = 'unetpp'
ENCODER      = 'resnet50'
USE_AMP      = True
AMP_DTYPE    = 'float16'

LR           = 1e-4
WEIGHT_DECAY = 1e-4            # same as every Forest Inspection run
EPOCHS       = 50              # fixed length, no early stopping: the FINAL model is evaluated,
                               # so no checkpoint is selected on the images that are reported
BATCH_SIZE   = 4
IMG_SIZE     = (768, 1024)     # (H, W); WildUAV Mapping frames are 5280x3956 (4:3)
MAX_HOURS    = 11.0            # do not start another loss if it would not finish before this
EVAL_EVERY   = 10              # print test mIoU every N epochs - monitoring only, never used to pick a model

SMOKE        = False           # True: 2 epochs on 16 training images per loss, to check the notebook (NOT for the paper)
MULTI_GPU    = True            # T4 x2: each batch of 4 is split over both GPUs (nn.DataParallel).
                               # Use the same GPU type for WU1-WU4 so that all runs share one setting.

# Output of an earlier session of THIS notebook (add it as an input dataset): finished losses are skipped.
DONE_DIR     = ''

WILD_UAV_OLD = '/kaggle/input/datasets/dinhh04/wilduav/WildUAV'
WILD_UAV_NEW = '/kaggle/input/datasets/phmnguynkimquyn/wuav-data'
TRAIN_TXT    = '/kaggle/input/datasets/itmeagain77/wilduav-splits/train.txt'
VAL_TXT      = '/kaggle/input/datasets/itmeagain77/wilduav-splits/val.txt'
ENCODER_WEIGHT_DIR = '/kaggle/input/datasets/itmeagain77/resnet50-pytorch-weights'
print(f'[CONFIG] {NOTEBOOK} | losses={LOSSES} | seed={SEED} | {IMG_SIZE} | wd={WEIGHT_DECAY} | SMOKE={SMOKE}')

[CONFIG] WU3_wilduav_s2026 | losses=['ce', 'ced', 'cefd', 'adaptive'] | seed=2026 | (768, 1024) | wd=0.0001 | SMOKE=False


In [3]:
import os
import numpy as np

# Official partition: train.txt (998 frames) for training, val.txt (246 frames) for reporting.
# Paths follow the earlier WildUAV notebook (images: seq00 in the old dataset, others in the new one).
def read_split_file(txt_path):
    pairs = []
    for line in open(txt_path):
        line = line.strip()
        if not line:
            continue
        seq, fname = line.split('/')
        base = fname.split('.')[0]
        if seq == 'seq00':
            img_dir = os.path.join(WILD_UAV_OLD, 'mapping_set', f'{seq}_img+metadata', 'img')
        else:
            img_dir = os.path.join(WILD_UAV_NEW, f'{seq}_imgmetadata.7z', f'{seq}_img+metadata.7z', 'img')
        img = next((p for p in (os.path.join(img_dir, base + e) for e in ('.png', '.jpg')) if os.path.exists(p)), None)
        lbl = next((p for p in (os.path.join(r, 'mapping_set', 'semantic_extension', f'{seq}_semantic', 'semantic', base + '.png')
                                for r in (WILD_UAV_OLD, WILD_UAV_NEW)) if os.path.exists(p)), None)
        if img and lbl:
            pairs.append((img, lbl))
    return pairs

train_pairs, test_pairs = read_split_file(TRAIN_TXT), read_split_file(VAL_TXT)
print(f'TRAIN {len(train_pairs)} | TEST (val.txt) {len(test_pairs)}')
assert (len(train_pairs), len(test_pairs)) == (998, 246), 'Unexpected split sizes - check the input dataset paths'
if SMOKE:
    train_pairs, test_pairs = train_pairs[:16], test_pairs[:8]

# 16 raw label colours -> the 9 benchmark classes (same mapping as the earlier notebook;
# unknown colours and rocks/building/fence/sky -> Empty). Empty is an active class, as in the official protocol.
WILDUAV_COLORS = [
    (0, 255, 255), (0, 127, 0), (19, 132, 69), (0, 53, 65), (130, 76, 0),
    (152, 251, 152), (151, 126, 171), (0, 0, 255), (250, 150, 0), (115, 176, 195),
    (128, 64, 128), (255, 77, 228), (123, 123, 123), (255, 255, 255), (200, 0, 0), (0, 0, 0),
]
CLASS_NAMES = ['Ground_vegetation', 'Trees', 'Dirt_ground', 'Road', 'Water', 'Empty',
               'Static_car', 'Moving_car', 'People']
NUM_CLASSES = len(CLASS_NAMES)
EMPTY_CLASS = 5
IGNORE = 255                   # only for border pixels created by the affine augmentation
RAW_TO_CLASS = np.full(16, EMPTY_CLASS, dtype=np.uint8)
for raw, cls in {0: 0, 5: 0, 1: 1, 2: 1, 3: 1, 4: 2, 10: 3, 11: 3, 7: 4, 15: 5, 12: 6, 13: 7, 14: 8}.items():
    RAW_TO_CLASS[raw] = cls
RGB_TO_CLASS = np.full((256, 256, 256), EMPTY_CLASS, dtype=np.uint8)
for raw, color in enumerate(WILDUAV_COLORS):
    RGB_TO_CLASS[color] = RAW_TO_CLASS[raw]


TRAIN 998 | TEST (val.txt) 246


In [4]:
# Decode and resize every frame once and keep it in RAM (about 4 GB): the 20-Mpixel PNGs are too slow
# to decode in every epoch. Image: area interpolation; label: nearest neighbour on the colour image.
import cv2
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

H, W = IMG_SIZE

def load_pair(pair):
    img = cv2.imread(pair[0], cv2.IMREAD_COLOR)
    lbl = cv2.imread(pair[1], cv2.IMREAD_COLOR)
    assert img is not None and lbl is not None, pair
    assert img.shape[:2] == lbl.shape[:2], f'image/label size mismatch: {pair}'
    shape = img.shape[:2]
    img = cv2.cvtColor(cv2.resize(img, (W, H), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    lbl = cv2.cvtColor(cv2.resize(lbl, (W, H), interpolation=cv2.INTER_NEAREST), cv2.COLOR_BGR2RGB)
    return img, RGB_TO_CLASS[lbl[..., 0], lbl[..., 1], lbl[..., 2]], shape

def load_all(pairs, desc):
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as ex:
        out = list(tqdm(ex.map(load_pair, pairs), total=len(pairs), desc=desc))
    return [o[0] for o in out], [o[1] for o in out], [o[2] for o in out]

train_imgs, train_masks, shapes = load_all(train_pairs, 'cache train')
test_imgs, test_masks, shapes_t = load_all(test_pairs, 'cache test')
aspects = {round(w / h, 3) for h, w in shapes + shapes_t}
print('native sizes:', sorted({f'{w}x{h}' for h, w in shapes + shapes_t}), '| aspect', aspects, '| resized to', f'{W}x{H}')
if any(abs(a - W / H) > 0.03 for a in aspects):
    print('[WARN] IMG_SIZE does not match the native aspect ratio - images are stretched')

counts = sum(np.bincount(m.ravel(), minlength=NUM_CLASSES)[:NUM_CLASSES] for m in train_masks)
freq = counts / counts.sum()
for n, f in zip(CLASS_NAMES, freq):
    print(f'  {n:<18}{100 * f:8.3f}%')


cache test: 100%|██████████| 246/246 [00:14<00:00, 16.64it/s]


native sizes: ['5280x3956'] | aspect {1.335} | resized to 1024x768
  Ground_vegetation   64.822%
  Trees               28.598%
  Dirt_ground          3.461%
  Road                 1.499%
  Water                1.400%
  Empty                0.139%
  Static_car           0.059%
  Moving_car           0.011%
  People               0.011%


In [5]:
import random
from pathlib import Path
import torch

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

AMP_DTYPE_T = torch.bfloat16 if AMP_DTYPE == 'bfloat16' else torch.float16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('[INFO] Device:', torch.cuda.get_device_name() if DEVICE == 'cuda' else 'cpu')

OUTPUT_DIR = Path('/kaggle/working/outputs') if os.path.exists('/kaggle') else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ENet weights w = 1/ln(1.02 + f_c) on the WildUAV training split, normalised to sum to NUM_CLASSES
w = 1.0 / np.log(1.02 + freq)
class_weights = torch.tensor(w / w.sum() * NUM_CLASSES, dtype=torch.float32).to(DEVICE)
print('[ENet weights]', dict(zip(CLASS_NAMES, np.round(class_weights.cpu().numpy(), 4))))


[INFO] Device: NVIDIA RTX PRO 6000 Blackwell Server Edition
[ENet weights] {'Ground_vegetation': np.float32(0.0628), 'Trees': np.float32(0.1203), 'Dirt_ground': np.float32(0.604), 'Road': np.float32(0.9338), 'Water': np.float32(0.9606), 'Empty': np.float32(1.5175), 'Static_car': np.float32(1.5754), 'Moving_car': np.float32(1.6126), 'People': np.float32(1.6131)}


In [6]:
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class CachedDataset(Dataset):
    def __init__(self, imgs, masks, transform):
        self.imgs, self.masks, self.transform = imgs, masks, transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        aug = self.transform(image=self.imgs[idx], mask=self.masks[idx])
        return aug['image'], aug['mask'].long()

# Same augmentation as Forest Inspection (images are already resized). Empty is an active class here,
# so the border pixels created by the affine transform are ignored instead of being labelled Empty.
train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), rotate=(-15, 15),
             translate_percent={'x': (-0.0625, 0.0625), 'y': (-0.0625, 0.0625)},
             p=0.3, border_mode=0, fill=0, fill_mask=IGNORE),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
test_tf = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

NUM_WORKERS = min(8, os.cpu_count() or 2)
train_ds = CachedDataset(train_imgs, train_masks, train_tf)
test_loader = DataLoader(CachedDataset(test_imgs, test_masks, test_tf), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)


/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


In [7]:
import shutil, glob
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

CACHE_DIR = '/root/.cache/torch/hub/checkpoints/'
os.makedirs(CACHE_DIR, exist_ok=True)
pth_files = glob.glob(os.path.join(ENCODER_WEIGHT_DIR, '*.pth'))
if pth_files and not os.path.exists(os.path.join(CACHE_DIR, 'resnet50-19c8e357.pth')):
    shutil.copy(pth_files[0], os.path.join(CACHE_DIR, 'resnet50-19c8e357.pth'))

N_GPUS = torch.cuda.device_count() if MULTI_GPU else 1

def build_model():
    m = smp.UnetPlusPlus(encoder_name=ENCODER, encoder_weights='imagenet',
                         in_channels=3, classes=NUM_CLASSES).to(DEVICE)
    return nn.DataParallel(m) if N_GPUS > 1 else m

print(f'[GPU] {N_GPUS} x', torch.cuda.get_device_name() if DEVICE == 'cuda' else 'cpu')

# ---- losses: same definitions as on Forest Inspection, but all 9 classes are active and only
# augmentation borders (IGNORE) are excluded from every term.
def make_ce():
    return nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE)

def make_focal():
    return smp.losses.FocalLoss(mode=smp.losses.MULTICLASS_MODE, gamma=2.0, ignore_index=IGNORE)

def make_dice():
    return smp.losses.DiceLoss(mode=smp.losses.MULTICLASS_MODE, from_logits=True, ignore_index=IGNORE)

class FixedWeightLoss(nn.Module):
    """Exp1 (ce), Exp2-CED, Exp2-CEFD (1:1:1): static weights."""
    def __init__(self, w_ce=1.0, w_focal=0.0, w_dice=0.0):
        super().__init__()
        self.w = {'ce': w_ce, 'focal': w_focal, 'dice': w_dice}
        self.ce = make_ce()
        self.focal = make_focal() if w_focal else None
        self.dice = make_dice() if w_dice else None

    def forward(self, logits, targets):
        l = {'ce': self.ce(logits, targets)}
        l['focal'] = self.focal(logits, targets) if self.focal else torch.zeros((), device=logits.device)
        l['dice'] = self.dice(logits, targets) if self.dice else torch.zeros((), device=logits.device)
        loss = sum(self.w[k] * l[k] for k in l)
        return loss, {**{k: v.item() for k, v in l.items()}, **{f'w_{k}': self.w[k] for k in l}}

    def get_weights(self):
        return {f'w_{k}': v for k, v in self.w.items()}

class AdaptiveFocalDiceLoss(nn.Module):
    """Exp3: homoscedastic uncertainty weighting. Logged w_* = exp(-s_k); effective weight = 0.5*exp(-s_k)."""
    def __init__(self):
        super().__init__()
        self.focal, self.dice, self.ce = make_focal(), make_dice(), make_ce()
        self.log_sigma_focal = nn.Parameter(torch.zeros(1))
        self.log_sigma_dice = nn.Parameter(torch.zeros(1))
        self.log_sigma_ce = nn.Parameter(torch.zeros(1))

    def forward(self, logits, targets):
        l_f, l_d, l_c = self.focal(logits, targets), self.dice(logits, targets), self.ce(logits, targets)
        w_f, w_d, w_c = (torch.exp(-s) for s in (self.log_sigma_focal, self.log_sigma_dice, self.log_sigma_ce))
        loss = (0.5 * w_f * l_f + 0.5 * self.log_sigma_focal
                + 0.5 * w_d * l_d + 0.5 * self.log_sigma_dice
                + 0.5 * w_c * l_c + 0.5 * self.log_sigma_ce)
        return loss.squeeze(), {'focal': l_f.item(), 'dice': l_d.item(), 'ce': l_c.item(),
                                'w_focal': w_f.item(), 'w_dice': w_d.item(), 'w_ce': w_c.item()}

    def get_weights(self):
        return {'w_focal': torch.exp(-self.log_sigma_focal).item(),
                'w_dice': torch.exp(-self.log_sigma_dice).item(),
                'w_ce': torch.exp(-self.log_sigma_ce).item()}

class UnifiedFocalLoss(nn.Module):
    """Symmetric Unified Focal loss (Yeung et al., 2022), lam=0.5, delta=0.6, gamma=0.5, as on Forest Inspection."""
    def __init__(self, lam=0.5, delta=0.6, gamma=0.5, eps=1e-7):
        super().__init__()
        self.lam, self.delta, self.gamma, self.eps = lam, delta, gamma, eps

    def forward(self, logits, targets):
        logits = logits.float()
        valid = targets != IGNORE
        prob = F.softmax(logits, dim=1).clamp(self.eps, 1 - self.eps)
        tgt = targets.clone()
        tgt[~valid] = 0
        p_t = prob.gather(1, tgt.unsqueeze(1)).squeeze(1)[valid]
        l_focal = ((1 - p_t) ** self.gamma * -torch.log(p_t)).mean()

        onehot = F.one_hot(tgt, NUM_CLASSES).permute(0, 3, 1, 2).float() * valid.unsqueeze(1)
        p = prob * valid.unsqueeze(1)
        dims = (0, 2, 3)
        tp, fn, fp = (p * onehot).sum(dims), ((1 - p) * onehot).sum(dims), (p * (1 - onehot)).sum(dims)
        ti = (tp + self.eps) / (tp + self.delta * fn + (1 - self.delta) * fp + self.eps)
        present = onehot.sum(dims) > 0
        l_ft = ((1 - ti[present]) ** (1 - self.gamma)).mean()

        loss = self.lam * l_ft + (1 - self.lam) * l_focal
        return loss, {'focal': l_focal.item(), 'dice': l_ft.item(), 'ce': 0.0,
                      'w_focal': 1 - self.lam, 'w_dice': self.lam, 'w_ce': 0.0}

    def get_weights(self):
        return {'w_focal': 1 - self.lam, 'w_dice': self.lam, 'w_ce': 0.0}

def build_criterion(name):
    if name == 'ce':            return FixedWeightLoss(1.0, 0.0, 0.0)
    if name == 'ced':           return FixedWeightLoss(1.0, 0.0, 1.0)
    if name == 'cefd':          return FixedWeightLoss(1.0, 1.0, 1.0)
    if name == 'adaptive':      return AdaptiveFocalDiceLoss()
    if name == 'unified_focal': return UnifiedFocalLoss()
    raise ValueError(name)

def confusion(model_, loader):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for imgs, masks in loader:
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE_T, enabled=USE_AMP):
                out = model_(imgs.to(DEVICE))
            p = out.argmax(1).cpu().numpy().ravel()
            t = masks.numpy().ravel()
            keep = t != IGNORE
            cm += np.bincount(t[keep] * NUM_CLASSES + p[keep], minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    return cm

def metrics_from_cm(cm):
    """Per-class IoU/recall/precision; mIoU over all 9 classes including Empty (benchmark protocol)."""
    tp = np.diag(cm).astype(float)
    gt, pred = cm.sum(1).astype(float), cm.sum(0).astype(float)
    iou = np.divide(tp, gt + pred - tp, out=np.zeros_like(tp), where=(gt + pred - tp) > 0)
    rec = np.divide(tp, gt, out=np.zeros_like(tp), where=gt > 0)
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    return float(iou.mean()), iou, rec, prec


[GPU] 1 x NVIDIA RTX PRO 6000 Blackwell Server Edition


In [8]:
# ============================================================
# TRAIN + TEST, one loss after another. Each run re-seeds, so its result does not depend on the order.
# ============================================================
import csv, json, gc

def train_one(loss_name):
    run = f'WU_{MODEL_NAME}_{loss_name}_s{SEED}' + ('_SMOKE' if SMOKE else '')
    seed_everything(SEED)
    model = build_model()
    criterion = build_criterion(loss_name).to(DEVICE)
    optimizer = torch.optim.AdamW(list(model.parameters()) + list(criterion.parameters()),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    epochs = 2 if SMOKE else EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE_T == torch.float16)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              pin_memory=True, drop_last=True, worker_init_fn=worker_init_fn,
                              generator=torch.Generator().manual_seed(SEED))
    log_file = OUTPUT_DIR / f'{run}_training_log.csv'
    with open(log_file, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'w_focal', 'w_dice', 'w_ce', 'l_focal', 'l_dice', 'l_ce', 'seconds'])
    print(f'\n===== {run} | {type(criterion).__name__} =====')
    t_run = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train(); criterion.train()
        sums = {k: 0.0 for k in ('focal', 'dice', 'ce', 'w_focal', 'w_dice', 'w_ce')}
        total, n, skipped = 0.0, 0, 0
        for imgs, masks in tqdm(train_loader, desc=f'E{epoch}', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE_T, enabled=USE_AMP):
                loss, comp = criterion(model(imgs), masks)
            if not torch.isfinite(loss):
                skipped += 1
                if skipped > 20:
                    raise RuntimeError(f'{skipped} non-finite losses in epoch {epoch} of {run}')
                continue
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            total += loss.item(); n += 1
            for k in sums:
                sums[k] += comp[k]
        scheduler.step()
        avg = {k: v / max(n, 1) for k, v in sums.items()}
        with open(log_file, 'a', newline='') as f:
            csv.writer(f).writerow([epoch, total / max(n, 1), avg['w_focal'], avg['w_dice'], avg['w_ce'],
                                    avg['focal'], avg['dice'], avg['ce'], round(time.time() - t0)])
        msg = f'E{epoch:02d} | loss {total / max(n, 1):.4f} | w F/D/CE {avg["w_focal"]:.2f}/{avg["w_dice"]:.2f}/{avg["w_ce"]:.2f} | {time.time() - t0:.0f}s'
        if skipped:
            msg += f' | skipped {skipped}'
        if epoch == 1 and DEVICE == 'cuda':
            msg += ' | peak GB ' + '/'.join(f'{torch.cuda.max_memory_allocated(d) / 1e9:.1f}' for d in range(max(N_GPUS, 1)))
        if epoch % EVAL_EVERY == 0 and epoch < epochs:
            model.eval()
            msg += f' | monitor mIoU {metrics_from_cm(confusion(model, test_loader))[0]:.4f}'
        print(msg)

    model.eval(); criterion.eval()
    cm = confusion(model, test_loader)
    miou, iou, rec, prec = metrics_from_cm(cm)
    results = {
        'run_name': run, 'dataset': 'WildUAV (train.txt -> val.txt)', 'model': MODEL_NAME, 'encoder': ENCODER,
        'loss': loss_name, 'seed': SEED, 'img_size': list(IMG_SIZE), 'epochs': epochs, 'selection': 'final epoch',
        'weight_decay': WEIGHT_DECAY, 'lr': LR, 'amp_dtype': str(AMP_DTYPE_T), 'smoke': SMOKE,
        'gpu': torch.cuda.get_device_name() if DEVICE == 'cuda' else 'cpu', 'n_gpus': N_GPUS,
        'train_minutes': round((time.time() - t_run) / 60, 1),
        'class_weights': dict(zip(CLASS_NAMES, class_weights.cpu().tolist())),
        'loss_weights_final': criterion.get_weights(),
        'test': {'miou': miou,
                 'iou': dict(zip(CLASS_NAMES, iou.tolist())),
                 'recall': dict(zip(CLASS_NAMES, rec.tolist())),
                 'precision': dict(zip(CLASS_NAMES, prec.tolist())),
                 'confusion_matrix': cm.tolist()},
    }
    json.dump(results, open(OUTPUT_DIR / f'results_{run}.json', 'w'), indent=2)
    torch.save({'run_name': run, 'model_state_dict': getattr(model, 'module', model).state_dict(),
                'criterion_state_dict': criterion.state_dict()}, OUTPUT_DIR / f'{run}_final.pth')
    print(f'TEST mIoU (9 classes) {miou:.4f} | {results["train_minutes"]} min')
    print(f"{'Class':<20}{'IoU':>8}{'Recall':>8}{'Prec':>8}")
    for c, a, b, d in zip(CLASS_NAMES, iou, rec, prec):
        print(f'{c:<20}{a:8.4f}{b:8.4f}{d:8.4f}')
    del model, criterion, optimizer, train_loader
    gc.collect(); torch.cuda.empty_cache()
    return time.time() - t_run

last_duration = 0.0
for loss_name in LOSSES:
    run = f'WU_{MODEL_NAME}_{loss_name}_s{SEED}' + ('_SMOKE' if SMOKE else '')
    done = sorted(Path(DONE_DIR).rglob(f'results_{run}.json')) if DONE_DIR else []
    if done:
        for p in Path(done[0]).parent.glob(f'*{run}*'):
            if p.suffix in ('.json', '.csv'):
                shutil.copy(p, OUTPUT_DIR / p.name)
        print(f'[SKIP] {run}: already finished in {done[0].parent}')
        continue
    if (time.time() - SESSION_START) + 1.1 * last_duration > MAX_HOURS * 3600:
        print(f'[TIME LIMIT] not starting {run}. Save this version, add its output as a dataset, set DONE_DIR and run again.')
        break
    last_duration = train_one(loss_name)
print('ALL DONE:', sorted(p.name for p in OUTPUT_DIR.glob('results_*.json')))


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/smp-hub/resnet50.imagenet/resolve/00cb74e366966d59cd9a35af57e618af9f88efe9/config.json
Retrying in 1s [Retry 1/5].
/usr/local/lib/python3.12/dist-packages/segmentation_models_pytorch/encoders/__init__.py:136: UserWarning: Error loading resnet50 `imagenet` weights from Hugging Face Hub, trying loading from original url...
  warnings.warn(message, UserWarning)



===== WU_unetpp_ce_s2026 | FixedWeightLoss =====


E01 | loss 1.0890 | w F/D/CE 0.00/0.00/1.00 | 56s | peak GB 11.6


E02 | loss 0.5580 | w F/D/CE 0.00/0.00/1.00 | 53s


E03 | loss 0.3628 | w F/D/CE 0.00/0.00/1.00 | 53s


E04 | loss 0.2778 | w F/D/CE 0.00/0.00/1.00 | 53s


... (202 outputs left out)

E48 | loss -0.8584 | w F/D/CE 1.93/1.95/1.91 | 57s


E49 | loss -0.8597 | w F/D/CE 1.93/1.95/1.91 | 57s


E50 | loss -0.8567 | w F/D/CE 1.93/1.95/1.91 | 57s


TEST mIoU (9 classes) 0.8782 | 48.0 min
Class                    IoU  Recall    Prec
Ground_vegetation     0.9549  0.9745  0.9794
Trees                 0.9147  0.9541  0.9568
Dirt_ground           0.8568  0.9688  0.8812
Road                  0.9523  0.9860  0.9654
Water                 0.9599  0.9903  0.9691
Empty                 0.8234  0.8963  0.9101
Static_car            0.9274  0.9739  0.9511
Moving_car            0.7528  0.8768  0.8418
People                0.7614  0.9105  0.8229
ALL DONE: ['results_WU_unetpp_adaptive_s2026.json', 'results_WU_unetpp_ce_s2026.json', 'results_WU_unetpp_ced_s2026.json', 'results_WU_unetpp_cefd_s2026.json']


In [9]:
# ============================================================
# DOWNLOAD — one zip with every results JSON and training log of this notebook
# ============================================================
import zipfile
WORK_DIR = OUTPUT_DIR.parent
zip_path = WORK_DIR / f'{NOTEBOOK}_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob('*'):
        if p.suffix in ('.json', '.csv'):
            z.write(p, arcname=p.name)
print(f'{zip_path.name}: {zip_path.stat().st_size / 1e3:.0f} kB (the *_final.pth files stay in outputs/)')
os.chdir(WORK_DIR)
try:
    from IPython.display import FileLink, display
    display(FileLink(zip_path.name))
except ImportError:
    print(zip_path)


WU3_wilduav_s2026_results.zip: 14 kB (the *_final.pth files stay in outputs/)


/kaggle/working/WU3_wilduav_s2026_results.zip